In [15]:
import pandas as pd

df_cust=pd.read_csv('data/customers.csv')
df_subs=pd.read_csv('data/subscriptions.csv')

df_payments=pd.read_csv('data/payments.csv')
df_marketing = pd.read_csv('data/marketing_spend.csv')

df_master = pd.merge(df_cust, df_subs, on='customer_id', how='left')

df_master['signup_date'] = pd.to_datetime(df_master['signup_date'])
df_master['start_date'] = pd.to_datetime(df_master['start_date'])
df_master['end_month'] = pd.to_datetime(df_master['end_date']).dt.to_period('M')

monthly_churn = df_master[df_master['end_date'].notnull()] \
    .groupby('end_month')['customer_id'].nunique()

print(monthly_churn)

print(df_master.head())

#Core SaaS Metrics

df_master['mrr'] = df_master['monthly_price'] * df_master['seats']
df_master['arr'] = df_master['mrr'] * 12


df_active = df_master[df_master['end_date'].isna()].copy()

total_mrr = df_active['mrr'].sum()
total_arr = df_active['arr'].sum()
active_customer_count = df_active['customer_id'].nunique()
arpu = total_mrr / active_customer_count if active_customer_count > 0 else 0

print("\n--- SaaS Core Metrics Summary ---")
print(f"Total Active Customers: {active_customer_count}")
print(f"Monthly Recurring Revenue (MRR): ${total_mrr:,.2f}")
print(f"Annual Recurring Revenue (ARR): ${total_arr:,.2f}")
print(f"Average Revenue Per User (ARPU): ${arpu:,.2f}")

mrr_by_plan = df_active.groupby('plan_type')['mrr'].sum()
print("\n--- Plan Based MRR Distribution ---")
print(mrr_by_plan)

end_month
2024-03     1
2024-05     1
2024-06     1
2024-07     3
2024-08     4
2024-09     5
2024-10     8
2024-11     7
2024-12     1
2025-01     7
2025-02     6
2025-03    11
2025-04     4
2025-05     8
2025-06    10
Freq: M, Name: customer_id, dtype: int64
   customer_id signup_date company_size       industry  country  \
0            1  2024-04-24         1-10      Education  Germany   
1            2  2024-08-16        11-50      Education      USA   
2            3  2025-03-08         1-10           Tech      USA   
3            4  2024-08-26         1-10  Manufacturing       UK   
4            5  2024-08-13         200+  Manufacturing  Germany   

  acquisition_channel   plan_type start_date end_date  monthly_price  seats  \
0            LinkedIn         Pro 2024-04-24      NaN             39     23   
1          Google Ads         Pro 2024-08-16      NaN             39     48   
2            LinkedIn       Basic 2025-03-08      NaN             19      9   
3            Referra

In [ ]:
# ---  Churn Analysis ---


lost_customers = df_master[df_master['end_date'].notnull()]['customer_id'].nunique()
total_customers = df_master['customer_id'].nunique()

overall_churn_rate = (lost_customers / total_customers) * 100


plan_churn = df_master.groupby('plan_type').apply(
    lambda x: x[x['end_date'].notnull()]['customer_id'].nunique()
    / x['customer_id'].nunique() * 100
)

print(f"\n--- Churn Analysis ---")
print(f"Overall Customer Churn Rate: %{overall_churn_rate:.2f}")
print("\n--- Plan Based Churn Rates ---")
print(plan_churn)

industry_churn = df_master.groupby('industry').apply(
    lambda x: (x['end_date'].notnull().sum() / x['customer_id'].nunique()) * 100
).sort_values(ascending=False)

print("\n--- Industry Based Churn Rates (From Highest to Lowest) ---")
print(industry_churn)


--- Churn Analysis ---
Overall Customer Churn Rate: %22.00

--- Plan Based Churn Rates ---
plan_type
Basic         24.277457
Enterprise     7.843137
Pro           24.603175
dtype: float64

--- Industry Based Churn Rates (From Highest to Lowest) ---
industry
Manufacturing    28.846154
Tech             26.229508
Retail           24.528302
Education        22.368421
Finance          17.543860
Healthcare       11.764706
dtype: float64


/var/folders/73/7m_fc6rx7k7cfh_j6tx4883c0000gn/T/ipykernel_41805/3060221640.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  plan_churn = df_master.groupby('plan_type').apply(
/var/folders/73/7m_fc6rx7k7cfh_j6tx4883c0000gn/T/ipykernel_41805/3060221640.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  industry_churn = df_master.groupby('industry').apply(


In [17]:
# LTV & CAC Calculation 


plan_metrics = df_active.groupby('plan_type').agg(
    mrr_sum=('mrr', 'sum'),
    customer_count=('customer_id', 'nunique')
)
plan_metrics['arpu'] = plan_metrics['mrr_sum'] / plan_metrics['customer_count']

plan_metrics['churn_rate'] = plan_churn / 100

plan_metrics['ltv'] = plan_metrics['arpu'] / plan_metrics['churn_rate']

print("\n--- Plan Based LTV (Lifetime Value) ---")
print(plan_metrics[['arpu', 'churn_rate', 'ltv']])


df_marketing = pd.read_csv('data/marketing_spend.csv') 

marketing_summary = df_marketing.groupby('channel').agg(
    total_spend=('spend', 'sum'),
    total_acquired=('customers_acquired', 'sum')
)
marketing_summary['cac'] = marketing_summary['total_spend'] / marketing_summary['total_acquired']


avg_cac = marketing_summary['cac'].mean()

plan_metrics['ltv_cac_ratio'] = plan_metrics['ltv'] / avg_cac
revenue_by_industry = df_active.groupby('industry')['mrr'].sum().sort_values(ascending=False)

print(revenue_by_industry)
df_active['start_month'] = df_active['start_date'].dt.to_period('M')

mrr_growth = df_active.groupby('start_month')['mrr'].sum()

print(mrr_growth)

print(plan_metrics[['ltv', 'ltv_cac_ratio']])

print("\n--- Channel Based CAC (Customer Acquisition Cost) ---")
print(marketing_summary.sort_values('cac'))


--- Plan Based LTV (Lifetime Value) ---
                   arpu  churn_rate           ltv
plan_type                                        
Basic        106.312977    0.242775    437.908215
Enterprise  2017.021277    0.078431  25717.021277
Pro         1125.252632    0.246032   4573.607470
industry
Education        61324
Healthcare       37678
Retail           31390
Tech             29813
Finance          28935
Manufacturing    26486
Name: mrr, dtype: int64
start_month
2024-01    16613
2024-02     9563
2024-03    12498
2024-04    16973
2024-05     8571
2024-06     8800
2024-07    10301
2024-08    17634
2024-09     7944
2024-10     5783
2024-11    12616
2024-12    15216
2025-01     6465
2025-02     6414
2025-03    11648
2025-04    23936
2025-05    11420
2025-06    13231
Freq: M, Name: mrr, dtype: int64
                     ltv  ltv_cac_ratio
plan_type                              
Basic         437.908215       1.163601
Enterprise  25717.021277      68.334743
Pro          4573.607470   

In [ ]:
# --- Cohort Retention ---


df_master['signup_month'] = df_master['signup_date'].dt.to_period('M')
df_master['end_date'] = pd.to_datetime(df_master['end_date'], errors='coerce')

today = pd.to_datetime('2026-04-16')
df_master['tenure_months'] = df_master.apply(
    lambda x: ((x['end_date'] if pd.notnull(x['end_date']) else today) - x['start_date']).days // 30,
    axis=1
)

print("\n--- Average Tenure by Plan ---")
print(df_master.groupby('plan_type')['tenure_months'].mean())


--- Ortalama Müşteri Ömrü (Ay Bazlı) ---
plan_type
Basic         14.427746
Enterprise    16.980392
Pro           14.841270
Name: tenure_months, dtype: float64


In [18]:
# ---  Revenue Leakage Analizi ---

df_payments = pd.read_csv('data/payments.csv')


payment_status = df_payments.groupby('status')['amount'].agg(['sum', 'count'])
payment_status['percentage_of_total_amount'] = (payment_status['sum'] / payment_status['sum'].sum()) * 100

print("\n--- Payment Status Distribution ---")
print(payment_status)


failed_payments_detailed = pd.merge(df_payments[df_payments['status'] == 'failed'], 
                                    df_master[['customer_id', 'plan_type']], 
                                    on='customer_id')
print("\n--- Plan Based Payment Failures ---")
print(failed_payments_detailed.groupby('plan_type').size())

failed_rev = df_payments[df_payments['status'] == 'failed']['amount'].sum()
refunded_rev = df_payments[df_payments['status'] == 'refunded']['amount'].sum()

print(f"\nFailed Payments Loss: ${failed_rev:,.2f}")
print(f"Refunded Payments Loss: ${refunded_rev:,.2f}")


--- Payment Status Distribution ---
              sum  count  percentage_of_total_amount
status                                              
failed     155742    203                    6.796052
paid      2069741   2626                   90.316470
refunded    66171     84                    2.887478

--- Plan Based Payment Failures ---
plan_type
Basic         95
Enterprise    33
Pro           75
dtype: int64

Failed Payments Loss: $155,742.00
Refunded Payments Loss: $66,171.00
